In [1]:
import os
import tensorly as tl
random_state=42

os.environ["TENSORLY_BACKEND"] = 'cupy'
tl.set_backend('cupy')

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation



paradigm = P300(resample=48)
dataset = BNCI2014_008()


<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.


To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(


In [3]:
hoda_params = dict(
    max_iter=128,
    tol=1e-12,
    init ='eye',
    shrinkage='lw',
    toeplitz=None,
    obj='rt',
    solver='lanczos',
    taper=False,
    extra_train_info=False,
    verbose=False, 
    random_state=random_state,
    delta=None,

)

In [4]:

from hoda.hoda import HODA,BTTDA
from sklearn.model_selection import GridSearchCV, cross_val_score
from hoda.tensorize import Tensorize, Vectorize
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

In [5]:
from hoda.hoda import BTTDA, GreedyBTTDA
from hoda.classification import SelectF
max_blocks = 2

pipe = Pipeline([
    ('bttda', GreedyBTTDA(
        max_blocks=max_blocks,
        hoda_params=dict(
            **hoda_params,
        ),
        extra_train_info=False,
        verbose=False,
        n_jobs=-1,
    )),
    ('clf', GreedyBTTDA.clf())
])

In [6]:
import math
from sklearn.metrics import roc_auc_score
import pandas as pd
import pdb

block_results = []
for subject in dataset.subject_list[:1]:
    #TODO: session
    epochs, labels, meta = paradigm.get_data(
        dataset=dataset, 
        subjects=[subject],
        return_epochs=True
    )
    sessions = meta['session'].unique()
    for session in sessions[:1]:
        print(f'subject={subject}, session={session}')
        ses_epochs = epochs[f"session == '{session}'"]
        X = tl.tensor(ses_epochs.get_data())
        y = labels[meta['session'] == session]
        result = cross_validate(pipe,X,y, cv=cv, return_estimator=True, return_indices=True, n_jobs=4, verbose=True)
        for fold in range(cv.n_splits):
            bttda = result['estimator'][fold]['bttda']
            train_idc = result['indices']['train'][fold]
            test_idc = result['indices']['test'][fold]
            
            val_scores = bttda.val_scores_
            for n_blocks in range(1, max_blocks+1):
                Xt = bttda.transform(X, blocks=bttda.blocks_[:n_blocks])
                clf =  result['estimator'][fold]['clf']
                clf.fit(Xt[train_idc],y[train_idc])
                y_pred = clf.decision_function(Xt)
                roc_auc = roc_auc_score(y[test_idc],y_pred[test_idc])
                
                block_results.append(dict(
                    subject=subject,
                    session=session,
                    fold=fold,
                    n_blocks=n_blocks,
                    roc_auc=roc_auc,
                    split='test'
                ))
                block_results.append(dict(
                    subject=subject,
                    session=session,
                    fold=fold,
                    n_blocks=n_blocks,
                    roc_auc=val_scores[n_blocks-1],
                    split='val'
                ))
%time block_results = pd.DataFrame(block_results)

Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied
subject=1, session=0


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/base.py:354: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  X = mne.concatenate_epochs(X)
/tmp/ipykernel_43912/2473136263.py:18: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X = tl.tensor(ses_epochs.get_data())
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


CUDADriverError: CUDA_ERROR_OUT_OF_MEMORY: out of memory

In [ ]:
block_results

In [ ]:
df = block_results
df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('default')
sns.lineplot(data=df, x='n_blocks', y='roc_auc',
             #hue='fold',
             palette='tab10',
             #ci=None, 
             hue='split'
            )

#plt.axhline(res_hoda[0].mean(), color='red')